# Create 3 Gold Layer Tables

In [0]:
from pyspark.sql.functions import avg, sum, max, min, col

# Read the silver table
silver_df = spark.table("crypto.silver.slv_crypto")

# Market overview Table
gold_market_df = silver_df.agg(
    sum("market_cap").alias("total_market_cap"),
    sum("total_volume").alias("total_market_volume"),
    avg("price_change_percentage_24h").alias("avg_market_change_24h"),
    max("current_price").alias("highest_price"),
    min("current_price").alias("lowest_price")
)
gold_market_df.display()

gold_market_df.write.format("delta").mode("overwrite").saveAsTable("crypto.gold.tbl_market")

In [0]:
# top 10 cryptocurrencies
gold_top_crypto_df = silver_df.orderBy(
    col("market_cap").desc()
).limit(10)

gold_top_crypto_df.display()

gold_top_crypto_df.write.format("delta").mode("overwrite").saveAsTable("crypto.gold.tbl_top_crypto")


In [0]:

# Liquidity Metrics Table - How actively a coin is traded.
gold_liquidity_df = silver_df.withColumn(
    "volume_marketcap_ratio",
    col("total_volume") / col("market_cap")
)

gold_liquidity_df.display()

gold_liquidity_df.write.format("delta").mode("overwrite").saveAsTable("crypto.gold.tbl_liquidity")


In [0]:
%sql
--Top Cryptocurrencies by Market Cap
SELECT name as coin,current_price as price, market_cap, total_volume FROM crypto.gold.tbl_top_crypto
ORDER BY market_cap DESC